# Positional Encoding in Transformers

Self-Attention converts words into contextual embeddings in parallel.

However, Self-Attention alone cannot capture the order of words in a sentence.

For example:

- "Man killed lion"
- "Lion killed man"

would appear almost the same to pure Self-Attention, even though their meanings are completely different.

This is not a problem in RNNs because RNNs process words sequentially.

---

# Simple Idea: Use Position Numbers

One simple solution is to add positional information to the embeddings.

For example, we can add another dimension representing the position of each word.

![](./images/img15.png)

---

# Problems with Simple Counting-Based Position Encoding

## 1. Unbounded Values

If positions are represented using simple counting:

```text
0, 1, 2, 3, 4, ...
```

then for large sequences like books or long documents, the values can become extremely large.

Neural networks generally work better with values in a bounded range such as:

$$
[-1,1]
$$

because very large values can make backpropagation unstable.

---

## Attempted Solution: Normalize by Total Length

We could divide positions by the total sentence length.

Example:

Sentence 1:

```text
Hello world
```

Sentence 2:

```text
Hello world its me
```

Then:

- In sentence 1, the second position becomes:

$$
1
$$

- In sentence 2, the second position becomes:

$$
0.5
$$

This creates inconsistency because the same word position gets different values depending on sentence length.

---

## 2. Discrete Values

Simple counting produces discrete integer values.

Neural networks usually learn better with smooth continuous representations rather than sharp discrete jumps.

---

## 3. Relative Positions Cannot Be Captured Well

Simple counting does not naturally represent relationships like:

- distance between words
- repeating patterns
- relative positioning

which are important for understanding language structure.

---

# Solution: Use Periodic Continuous Functions

To solve these issues, Transformers use periodic trigonometric functions such as:

- Sine
- Cosine

---

# Why Sine and Cosine?

## 1. Bounded

Sine and cosine values always remain between:

$$
-1 \text{ and } 1
$$

which makes training stable.

---

## 2. Continuous

These functions are continuous, meaning values exist smoothly at every point.

This helps neural networks learn patterns more effectively.

---

## 3. Periodic

Their repeating nature helps encode relative positions and repeating structures in sequences.

---

# Understanding Positional Encoding with Sine and Cosine

Consider the sentence:

```text
Man killed a lion
```

We calculate positional values using the sine function:

$$
y = \sin(\text{position})
$$

So:

- "Man" → \(\sin(1)\)
- "killed" → \(\sin(2)\)
- "a" → \(\sin(3)\)
- "lion" → \(\sin(4)\)

Example values:

- \(\sin(1) \approx 0.84\)
- \(\sin(2) \approx 0.90\)

These values are appended to the embedding vectors before passing them into the Self-Attention layer.

---

# Why This Helps

Using sine-based positional values solves several earlier problems:

- Values remain bounded between \(-1\) and \(1\)
- The function is continuous
- Relative positions can be represented smoothly
- Better suited for neural networks

![](./images/img16.png)

---

# Problem with Using Only Sine

Sine is a periodic function.

This means different positions can eventually produce the same value.

For example:

$$
\sin(x) = \sin(x + 2\pi)
$$

So two different positions may end up with identical positional values.

This creates ambiguity.

---

# Solution: Use Both Sine and Cosine

Instead of using only sine, Transformers use both:

$$
y = \sin(\text{pos})
$$

and

$$
y = \cos(\text{pos})
$$

Both values are appended to the embedding vector.

![](./images/img17.png)

---

# Positional Encoding Becomes a Vector

Now positional encoding is no longer a single scalar value.

Instead, it becomes a vector of positional features.

Example:

```text
[
sin(pos),
cos(pos)
]
```

This significantly reduces ambiguity.

---

# Extending the Idea Further

If repetition still occurs, we can use multiple sine and cosine curves with different frequencies.

Example:

$$
y = \sin(\text{pos})
$$

$$
y = \cos(\text{pos})
$$

$$
y = \sin(\text{pos}/2)
$$

$$
y = \cos(\text{pos}/2)
$$

Using multiple periodic curves creates unique positional patterns for different positions.

This is the core idea behind the positional encoding used in Transformers.

# Positional Encoding in *Attention Is All You Need*

In the Transformer architecture proposed in *Attention Is All You Need*, a positional encoding vector is created for every word.

The dimension of the positional encoding vector is the same as the embedding dimension.

---

# Adding Positional Encoding

Instead of appending positional values to the embeddings, Transformers:

- create a positional encoding vector
- add it directly to the word embedding

The result is then passed into the Self-Attention block.

$$
\text{Input to Attention} =
\text{Word Embedding} +
\text{Positional Encoding}
$$

This allows the model to preserve word order information while still processing all words in parallel.

---

# Using Sine and Cosine Pairs

For every two dimensions of the positional encoding vector, one sine-cosine pair is used.

For example:

For the first two dimensions:

$$
PE(pos, 0) = \sin(pos)
$$

$$
PE(pos, 1) = \cos(pos)
$$

For the third and fourth dimensions:

$$
PE(pos, 2) = \sin(pos/2)
$$

$$
PE(pos, 3) = \cos(pos/2)
$$

and so on.

Different dimensions use different frequencies of sine and cosine waves.

---

# Original Positional Encoding Formula

The actual formula used in the paper is:

$$
PE(pos, 2i) =
\sin\left(
\frac{pos}{10000^{2i/d_{model}}}
\right)
$$

$$
PE(pos, 2i+1) =
\cos\left(
\frac{pos}{10000^{2i/d_{model}}}
\right)
$$

---

# Meaning of Terms

- \(pos\) → position of the word in the sequence
- \(d_{model}\) → dimension of the embedding vector
- \(i\) → dimension index ranging from:

$$
0 \text{ to } \frac{d_{model}}{2} - 1
$$

---

# Why Different Frequencies?

Different frequencies allow the model to capture:

- short-range relationships
- long-range relationships
- relative distances between words

Each dimension learns positional information at a different scale.

---

# Final Idea

By adding positional encoding vectors to embeddings, Transformers can understand sequence order without using recurrence like RNNs.

This enables efficient parallel computation while still preserving positional information.

# Observations About Positional Encoding

![](./images/img18.png)

---

## 1. Frequency Decreases Across Dimensions

We can observe that the frequency of the sine and cosine waves decreases for higher-dimensional pairs.

Lower dimensions have high-frequency oscillations, while higher dimensions change more slowly.

---

## 2. Alternating Pattern in Lower Dimensions

For the first positional encoding dimensions, we can see alternating white and blue patterns.

This represents rapidly changing values such as:

```text
0 1 0 1 0 1 ...
```

or quickly oscillating sine/cosine waves.

These dimensions capture fine-grained positional differences between nearby words.

---

## 3. More Variation in Lower Dimensions

Most of the visible variation appears in lower dimensions.

Higher dimensions look smoother because their frequencies are lower.

This happens because the denominator in the positional encoding formula increases with dimension:

$$
10000^{2i/d_{model}}
$$

As \(i\) increases:

- frequency decreases
- wavelength increases
- values change more gradually

---

# Intuition

Lower dimensions capture:

- local positional relationships
- short-distance changes

Higher dimensions capture:

- broader positional structure
- long-range relationships

Together, all dimensions provide unique positional information for every word in the sequence.